### RAG Pipeline - Data Ingestion to Vector DB 

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [7]:
### Read all pdfs inside the directory

def read_pdfs(pdf_directory):
    pdf_dir = Path(pdf_directory)
    pdf_loader = DirectoryLoader(
        pdf_dir,
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        silent_errors=True,
        show_progress=False,
        loader_kwargs={"extract_images": False}
    )
    all_documents = pdf_loader.load()

    for doc in all_documents:
        doc.metadata["file_type"] = "pdf"

    print(f"Read {len(all_documents)} documents from {pdf_directory}")

    return all_documents

In [8]:
documents = read_pdfs("../data")
documents

Read 68 documents from ../data


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-03T14:19:44+00:00', 'source': '../data/pdf/embeddings.pdf', 'file_path': '../data/pdf/embeddings.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-03T14:19:44+00:00', 'trapped': '', 'modDate': "D:20260503141944+00'00'", 'creationDate': "D:20260503141944+00'00'", 'page': 0, 'file_type': 'pdf'}, page_content="Embeddings in Machine Learning\nWhat Are Embeddings?\nEmbeddings are dense, low-dimensional vector representations of high-dimensional data\nsuch as words, sentences, images, or users. Instead of representing data as sparse one-hot\nvectors, embeddings map each item to a continuous point in a learned vector space.\nThe key insight is that semantically similar items end up close together in this space. For\nexample, the words 'king' and 'queen' will have 

### Text Splitting

In [9]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Chunk example
    if split_docs:
        print("Chunk example:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [10]:
chunks = split_documents(documents)
chunks

Split 68 documents into 267 chunks
Chunk example:
Content: Embeddings in Machine Learning
What Are Embeddings?
Embeddings are dense, low-dimensional vector representations of high-dimensional data
such as words, sentences, images, or users. Instead of represe...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-03T14:19:44+00:00', 'source': '../data/pdf/embeddings.pdf', 'file_path': '../data/pdf/embeddings.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-03T14:19:44+00:00', 'trapped': '', 'modDate': "D:20260503141944+00'00'", 'creationDate': "D:20260503141944+00'00'", 'page': 0, 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-03T14:19:44+00:00', 'source': '../data/pdf/embeddings.pdf', 'file_path': '../data/pdf/embeddings.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-03T14:19:44+00:00', 'trapped': '', 'modDate': "D:20260503141944+00'00'", 'creationDate': "D:20260503141944+00'00'", 'page': 0, 'file_type': 'pdf'}, page_content="Embeddings in Machine Learning\nWhat Are Embeddings?\nEmbeddings are dense, low-dimensional vector representations of high-dimensional data\nsuch as words, sentences, images, or users. Instead of representing data as sparse one-hot\nvectors, embeddings map each item to a continuous point in a learned vector space.\nThe key insight is that semantically similar items end up close together in this space. For\nexample, the words 'king' and 'queen' will have 

### Embedding and Vector DB

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [12]:
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="../data/vector_store"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5672.52it/s]


In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [14]:
results = retriever.invoke("What is attention?")

In [15]:
print(results)

[Document(id='59907586-9ea7-4f67-8ea9-0eff6c261b8e', metadata={'page': 2, 'title': '', 'creationDate': 'D:20240410211143Z', 'file_path': '../data/pdf/attention_is_all_you_need.pdf', 'source': '../data/pdf/attention_is_all_you_need.pdf', 'file_type': 'pdf', 'total_pages': 15, 'creator': 'LaTeX with hyperref', 'modDate': 'D:20240410211143Z', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'creationdate': '2024-04-10T21:11:43+00:00', 'producer': 'pdfTeX-1.40.25', 'subject': '', 'author': '', 'keywords': '', 'format': 'PDF 1.5'}, page_content='3.2\nAttention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3'), Document(id='a01fa112-4d5d-455f-84c0-6329809bf4af', metadata={'creator': 'LaTeX with hyperref', 'total_pages': 15, 'trapped': '', 'source': '../data/pdf/attention_is_all_you_need.pdf', 'creationDate': 'D:20240410211143Z', '